# 02. 텍스트 RAG 기본형 (v0)

PDF에서 **글자만** 추출해 만든 가장 기본적인 RAG입니다.

```
PDF → 글자 추출 → 청크 분할 → 임베딩 → FAISS 저장
질문 → 임베딩 → 비슷한 청크 5개 검색 → LLM 답변 + 근거(문서·쪽)
```

- v0는 이후 버전(v1 표 구조, v2 Vision 캡션)과 비교하는 **기준(baseline)**입니다.
- 차트 질문에 못 답하는 것이 정상이며, 그 실패가 개선 효과를 보여주는 근거가 됩니다.
- **API 사용 (요금 발생)** 표시가 있는 칸은 OpenAI API를 호출합니다. 전체 1회 실행 예상 비용: 약 $0.01

## ① 설정
`.env`에서 API 키와 모델 이름을 읽습니다. 키 값은 출력하지 않고 "있음/없음"만 표시합니다.

In [1]:
import os
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*langchain-community.*")  # 지원 종료 예정 경고 숨김 (5주차에 pgvector로 교체)

from dotenv import load_dotenv
import pandas as pd
import pymupdf
import tiktoken
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")

LLM_MODEL = os.getenv("LLM_MODEL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "800"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))
TOP_K = 5  # 검색해서 LLM에 넘길 청크 수

# 키 값은 절대 출력하지 않고, 있는지만 확인
print("API 키:", "있음" if os.getenv("OPENAI_API_KEY") else "없음 (.env 확인)")
print("LLM_MODEL:", LLM_MODEL)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("CHUNK_SIZE / OVERLAP:", CHUNK_SIZE, CHUNK_OVERLAP)

API 키: 있음
LLM_MODEL: gpt-6-luna
EMBEDDING_MODEL: text-embedding-3-small
CHUNK_SIZE / OVERLAP: 800 100


## ② 글자 추출
`01_pdf_explore`에서 정한 사용 범위의 쪽만, `sort=True`(위치순 정렬)로 추출합니다.
쪽마다 문서 ID와 쪽 번호를 붙여 두어, 나중에 답변의 근거로 표시합니다.

In [2]:
PDF_DIR = PROJECT_DIR / "data" / "pdf"
DOCS = {
    "D01": "2026_youth_employment_survey.pdf",
    "D02": "2025_content_industry_trend.pdf",
    "D03": "2025_social_survey.pdf",
}
# 01_pdf_explore에서 정한 사용 범위 (5주차에 app/config.py로 옮길 예정)
PAGE_RANGES = {
    "D01": [(1, 40)],
    "D02": [(13, 22), (37, 54)],
    "D03": [(1, 56)],
}

def pages_in_range(doc_id):
    for start, end in PAGE_RANGES[doc_id]:
        yield from range(start, end + 1)

def clean(text):
    """sort=True 추출로 생긴 연속 공백과 빈 줄 정리"""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

page_docs = []
for doc_id, file_name in DOCS.items():
    pdf = pymupdf.open(PDF_DIR / file_name)
    for page_no in pages_in_range(doc_id):
        text = clean(pdf[page_no - 1].get_text(sort=True))
        if len(text) < 50:  # 빈 쪽, 간지 제외
            continue
        # 쪽마다 문서 ID와 쪽 번호를 붙여 둠 → 나중에 답변 근거로 표시
        page_docs.append(Document(page_content=text, metadata={"doc_id": doc_id, "page": page_no}))

print("추출한 쪽 수:", len(page_docs))

추출한 쪽 수: 121


## ③ 청크 분할
쪽 단위 글을 `CHUNK_SIZE`(800자)씩, `CHUNK_OVERLAP`(100자)씩 겹치게 자릅니다.
30자 미만의 짧은 조각은 의미가 없어 제외합니다. 임베딩 비용도 미리 계산합니다.

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
# 쪽 단위로 자르므로 청크가 두 쪽에 걸치지 않음 → 근거 쪽수가 정확함
chunks = [c for c in splitter.split_documents(page_docs) if len(c.page_content) >= 30]

print("청크 수:", len(chunks))
print(pd.Series([c.metadata["doc_id"] for c in chunks]).value_counts().sort_index())

enc = tiktoken.get_encoding("cl100k_base")  # text-embedding-3-small이 쓰는 토큰 방식
embed_tokens = sum(len(enc.encode(c.page_content)) for c in chunks)
print(f"임베딩 토큰: {embed_tokens:,}개 → 예상 비용 ${embed_tokens / 1_000_000 * 0.02:.4f}")

청크 수: 264
D01     87
D02     50
D03    127
Name: count, dtype: int64
임베딩 토큰: 146,012개 → 예상 비용 $0.0029


## ④ 임베딩 + FAISS 저장 — API 사용 (요금 발생)
청크를 벡터로 바꿔 FAISS에 저장합니다. 한 번 만든 인덱스는 `data/index/v0_faiss/`에 저장되어,
다음 실행부터는 불러오기만 합니다. (임베딩 비용 0)

> 청크 설정(CHUNK_SIZE 등)을 바꿨다면 `data/index/v0_faiss/` 폴더를 지우고 다시 실행해야 새 설정이 반영됩니다.

In [4]:
INDEX_DIR = PROJECT_DIR / "data" / "index" / "v0_faiss"  # GitHub에 안 올라감
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

if INDEX_DIR.exists():
    # 직접 만든 파일이므로 불러와도 안전 (모르는 출처의 파일은 불러오면 안 됨)
    vectorstore = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
    print("저장된 인덱스를 불러왔습니다 (임베딩 비용 0)")
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(str(INDEX_DIR))
    print("인덱스를 새로 만들어 저장했습니다:", INDEX_DIR)

print("저장된 청크 수:", vectorstore.index.ntotal)

인덱스를 새로 만들어 저장했습니다: /home/user/git_project/project6/ReportQA/data/index/v0_faiss
저장된 청크 수: 264


## ⑤ 검색 시험 — API 사용 (질문 임베딩 1회)
질문과 가장 가까운 청크 5개를 찾습니다. 점수는 **거리**라서 **작을수록** 질문과 비슷합니다.

In [5]:
def search(question, k=TOP_K):
    """질문과 가장 가까운 청크 k개와 거리 점수를 돌려줌 (거리가 작을수록 비슷함)"""
    return vectorstore.similarity_search_with_score(question, k=k)

q = "2025년 연간 콘텐츠산업 매출액은 얼마인가?"
for doc, score in search(q):
    print(f"[{doc.metadata['doc_id']} p.{doc.metadata['page']}] 거리 {score:.3f} | {doc.page_content[:60]!r}")

[D02 p.50] 거리 0.498 | '나. 2025년 연간 콘텐츠산업 규모\n 1) 2025년 연간 매출액\n ❍ ’25년 연간 매출액 추정치는 약 '
[D02 p.18] 거리 0.511 | '2. 2025년 연간 콘텐츠산업 규모\n 가. 2025년 연간 매출액\n ❍ ’25년 연간 매출액 추정치는 약 '
[D02 p.15] 거리 0.536 | '2025년 4분기 콘텐츠산업전년, 동기 대비 매출종사자· 증가수출, 감소\n ▪ 매출액 45조 132억 원전년'
[D02 p.52] 거리 0.551 | '2) 2025년 연간 수출액\n ❍ ’25년 연간 수출액 추정치는 약 149억 582만 달러로 추정되었으며전년'
[D02 p.46] 거리 0.615 | '2) 2025년 4분기 수출액\n ❍ ’25년 4분기 추정치 산출 결과 전체 콘텐츠산업의 수출액은 약 45억 '


## ⑥ 답변 생성 — API 사용 (요금 발생)
검색된 청크만 근거로 LLM이 답하게 합니다. 근거를 `[D01 p.8]` 형식으로 표시하고, 근거에 없으면 "모름"으로 답하게 합니다.

In [6]:
llm = ChatOpenAI(model=LLM_MODEL)

PROMPT = """당신은 통계 보고서 내용으로만 답하는 도우미입니다.
규칙:
1. 아래 [근거]에 있는 내용으로만 답하세요. 추측하지 마세요.
2. 답의 끝에 사용한 근거를 [문서ID p.쪽] 형식으로 표시하세요. 예: [D01 p.8]
3. [근거]에서 답을 찾을 수 없으면 정확히 "모름"이라고만 답하세요.

[근거]
{context}

[질문]
{question}"""

usage = {"input": 0, "output": 0}  # 사용한 토큰 누적 (비용 계산용)

def answer(question):
    found = search(question)
    context = "\n\n".join(
        f"[{d.metadata['doc_id']} p.{d.metadata['page']}]\n{d.page_content}" for d, _ in found
    )
    response = llm.invoke(PROMPT.format(context=context, question=question))
    meta = response.usage_metadata or {}
    usage["input"] += meta.get("input_tokens", 0)
    usage["output"] += meta.get("output_tokens", 0)
    sources = [f"{d.metadata['doc_id']} p.{d.metadata['page']}" for d, _ in found]
    return response.text, sources

text, sources = answer(q)
print(text)
print("검색된 근거:", sources)

약 161조 4,839억 원입니다. [D02 p.18]
검색된 근거: ['D02 p.50', 'D02 p.18', 'D02 p.15', 'D02 p.52', 'D02 p.46']


## ⑦ 맛보기 질문 10개 — API 사용 (요금 발생)
텍스트 4 / 표 3 / 차트 3개 질문으로 v0가 **어떤 유형에서 실패하는지** 확인합니다.
정답 확인과 정식 평가(110문항, Recall@5)는 3주차 테스트셋에서 합니다.

- **정답 문서 검색됨:** 검색된 5개 청크 중에 정답이 있는 문서가 하나라도 있는지

In [7]:
# (유형, 정답이 있는 문서, 질문) — 정답 확인과 정식 평가는 3주차 테스트셋에서
QUESTIONS = [
    ("텍스트", "D03", "19세 이상 인구 중 노후 준비를 하고 있다고 응답한 비율은?"),
    ("텍스트", "D03", "고령자의 생활비 마련 방법 중 가장 많은 것은?"),
    ("텍스트", "D02", "2025년 연간 콘텐츠산업 수출액은 얼마인가?"),
    ("텍스트", "D01", "청년층 인구 중 최종학교 졸업자는 몇 명인가?"),
    ("표",     "D01", "2026년 5월 청년층 실업률은?"),
    ("표",     "D03", "향후 늘려야 할 공공시설 중 보건의료시설의 비율은?"),
    ("표",     "D02", "2025년 4분기 음악 산업 매출액은?"),
    ("차트",   "D01", "2026년 5월 20~24세의 고용률은?"),
    ("차트",   "D03", "고령자의 생활비를 본인·배우자가 부담하는 비율은 2015년 이후 어떻게 변했나?"),
    ("차트",   "D02", "2025년 4분기 장르별 매출액 그래프에서 매출액이 가장 큰 산업은?"),
]

rows = []
for qtype, doc_id, q in QUESTIONS:
    text, sources = answer(q)
    rows.append({
        "유형": qtype,
        "질문": q,
        "답변": text,
        "검색된 근거": ", ".join(sources),
        "정답 문서 검색됨": any(s.startswith(doc_id) for s in sources),
    })

pd.set_option("display.max_colwidth", None)
results = pd.DataFrame(rows)
results

,유형,질문,답변,검색된 근거,정답 문서 검색됨
0,텍스트,19세 이상 인구 중 노후 준비를 하고 있다고 응답한 비율은?,71.5%입니다. [D03 p.17],"D03 p.17, D03 p.8, D03 p.18, D03 p.19, D03 p.4",True
1,텍스트,고령자의 생활비 마련 방법 중 가장 많은 것은?,본인·배우자 부담(79.7%)입니다. [D03 p.21],"D03 p.8, D03 p.21, D03 p.21, D03 p.38, D03 p.17",True
2,텍스트,2025년 연간 콘텐츠산업 수출액은 얼마인가?,약 149억 582만 달러입니다. [D02 p.18],"D02 p.18, D02 p.52, D02 p.50, D02 p.15, D02 p.46",True
3,텍스트,청년층 인구 중 최종학교 졸업자는 몇 명인가?,400만 3천명입니다. [D01 p.10],"D01 p.11, D01 p.10, D01 p.17, D01 p.8, D01 p.6",True
4,표,2026년 5월 청년층 실업률은?,7.2%입니다. [D01 p.6],"D01 p.9, D01 p.36, D01 p.3, D01 p.6, D01 p.26",True
5,표,향후 늘려야 할 공공시설 중 보건의료시설의 비율은?,29.1% [D03 p.15],"D03 p.15, D03 p.15, D03 p.4, D03 p.37, D03 p.9",True
6,표,2025년 4분기 음악 산업 매출액은?,모름,"D02 p.15, D02 p.46, D02 p.44, D02 p.45, D02 p.50",True
7,차트,2026년 5월 20~24세의 고용률은?,모름,"D01 p.26, D01 p.36, D03 p.5, D03 p.39, D03 p.52",True
8,차트,고령자의 생활비를 본인·배우자가 부담하는 비율은 2015년 이후 어떻게 변했나?,"2015년 66.6%에서 2017년 69.9%, 2019년 69.9%, 2021년 72.5%, 2023년 76.0%, 2025년 79.7%로 전반적으로 증가했습니다. [D03 p.8] [D03 p.21]","D03 p.8, D03 p.21, D03 p.21, D03 p.49, D03 p.22",True
9,차트,2025년 4분기 장르별 매출액 그래프에서 매출액이 가장 큰 산업은?,지식정보입니다. [D02 p.44],"D02 p.46, D02 p.50, D02 p.44, D02 p.52, D02 p.15",True


## ⑧ 비용 기록
이번 실행에서 사용한 토큰과 비용을 계산합니다.

In [8]:
PRICE_PER_1M = {  # (입력, 출력) 달러, 2026-11-16 OpenAI 공식 가격
    "gpt-6-luna": (0.10, 0.50),
    "gpt-6.1-terra": (1.00, 5.00),
}
price_in, price_out = PRICE_PER_1M.get(LLM_MODEL, (0, 0))
llm_cost = usage["input"] / 1_000_000 * price_in + usage["output"] / 1_000_000 * price_out
embed_cost = embed_tokens / 1_000_000 * 0.02

print(f"LLM 토큰: 입력 {usage['input']:,} / 출력 {usage['output']:,} → ${llm_cost:.4f}")
print(f"임베딩 (인덱스를 새로 만든 경우만 발생): ${embed_cost:.4f}")

LLM 토큰: 입력 26,418 / 출력 554 → $0.0029
임베딩 (인덱스를 새로 만든 경우만 발생): $0.0029


## ⑨ 발견한 것

**결과:** 10문항 중 8개 정답, 2개 "모름" (지어낸 답 0개). 비용 약 $0.006

| 유형(당시 분류) | 결과 | 원인 / 메모 |
|---|---|---|
| 텍스트 | 4/4 정답 | 본문 문장에 답이 있으면 잘 찾음 |
| 표 | 2/3 정답 | 음악 매출액(D02 p.45 표 1-1): 정답 쪽은 검색됐지만 열 제목이 사라져 숫자 8개 중 어느 것이 '2025년 4분기'인지 알 수 없음 → **v1 표 구조 반영 필요** |
| 차트 | 2/3 정답 | 아래 정정 참고 |

### 정정 (2026-11-16, 3주차 테스트셋 작성 중 원문을 다시 확인한 결과)

처음에는 "차트" 질문 3개의 결과를 차트 읽기 능력으로 해석했지만, 원문을 확인해 보니 두 가지가 틀렸습니다.

1. **20~24세 고용률 ("모름") — 원인은 차트가 아니라 검색 실패**
   - 처음 분석: "D01 p.10 차트가 이미지 조각이라 글자 0개 → v2 Vision 필요"
   - 실제: 값(41.6%)은 **D01 p.9 본문 문장과 표에 글자로 있음**. 검색 결과 5개에 p.9가 없었음 (p.26, p.36, D03 쪽들이 검색됨)
   - → Vision이 아니라 **검색 개선(6주차 하이브리드 검색·리랭커)**으로 풀어야 할 실패
   - → 평가 가이드 기준으로 이 질문은 "차트"가 아니라 **"텍스트"** 유형

2. **장르별 매출 1위 (정답) — 차트가 아니라 본문 문장으로 맞힘**
   - 처음 분석: "D02 차트는 벡터라 숫자 라벨이 글자로 추출되어 v0도 답함"
   - 실제: D02 차트(그림 1-1)의 숫자는 **글자로 추출되지 않음**. 본문에 "지식정보의 매출액이 … 가장 큰 비중"이라는 문장이 있어 맞힌 것
   - → 이 질문도 **"텍스트"** 유형

3. **고령자 생활비 추이 (정답) — 분석 유지**
   - D03 차트는 벡터라 숫자 라벨이 글자로 추출됨 (순서는 뒤섞임). 2015~2021년 값은 차트에만 있어 **진짜 차트 질문**이 맞음

**정정 후 유형별 결과:** 텍스트 5/6, 표 2/3, 차트 1/1
- 이 10문항은 맛보기라 숫자 자체보다 "실패 원인 3가지(검색 실패, 표 구조 손실, 이미지 차트)"를 확인한 것이 의미 있음
- 정식 비교는 `eval/testset.csv` 110문항으로 함 (`notebooks/03_eval_v0.ipynb`)

**교훈 → 평가 가이드에 반영함**
- 질문 유형은 "질문에 '그래프'라는 말이 있는가"가 아니라 **"정답이 문서 어디에 있는가"**로 정해야 함
- "정답 문서 검색됨"은 문서가 3개뿐이라 너무 쉬운 기준 → **쪽 단위 Recall@5**로 측정